In [1]:
# CELL 1: GPU CHECK
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}, {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU count: 2
GPU 0: Tesla T4, 15.6 GB
GPU 1: Tesla T4, 15.6 GB


In [2]:
# CELL 2: INSTALL DEPENDENCIES
!pip install -q -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 101.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 47.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.1 MB/s eta 0:00:00


In [3]:
# CELL 3: IMPORTS AND CONFIG
import os, json, time, gc, re, math
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy.optimize import minimize_scalar

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN loaded.")
except Exception:
    hf_token = None
    print("No HF_TOKEN secret found — official gated repos will fail over to mirrors automatically.")

print("Setup ready.")
print("Checkpoints:", CHECKPOINT_DIR)
print("Outputs:", OUTPUT_DIR)

HF_TOKEN loaded.
Setup ready.
Checkpoints: /kaggle/working/checkpoints
Outputs: /kaggle/working/outputs


In [4]:
# CELL 3B: COPY PRE-EXISTING CYCLIC CHECKPOINTS FROM PREVIOUS RUN (SO CELLS 9-10B SKIP INSTANTLY)
import shutil

PREVIOUS_CHECKPOINT_SOURCE = "/kaggle/input/datasets/nagajaideepchowdary/cyclic-with-qwen/checkpoints"

if os.path.exists(PREVIOUS_CHECKPOINT_SOURCE):
    for fname in os.listdir(PREVIOUS_CHECKPOINT_SOURCE):
        if fname == "__huggingface_repos__.json":
            continue  # metadata file, not a game checkpoint — skip it
        src = os.path.join(PREVIOUS_CHECKPOINT_SOURCE, fname)
        dst = os.path.join(CHECKPOINT_DIR, fname)
        if not os.path.exists(dst):
            shutil.copy(src, dst)
            print(f"Copied: {fname}")
        else:
            print(f"Already present, skipped: {fname}")
else:
    print(f"WARNING: {PREVIOUS_CHECKPOINT_SOURCE} not found — Llama/R1/Qwen3 will re-run from scratch.")

Copied: DeepSeek-R1-Distill-Qwen-14B_cyclic_repeated.json
Copied: Qwen3-8B_cyclic_oneshot_samples.json
Copied: Qwen3-8B_cyclic_repeated.json
Copied: Llama-3-8B-Instruct_cyclic_repeated.json
Copied: DeepSeek-R1-Distill-Qwen-14B_cyclic_oneshot_samples.json
Copied: Llama-3-8B-Instruct_cyclic_oneshot_samples.json


In [5]:
# CELL 4: MODEL LOADER WITH OFFICIAL-FIRST, MIRROR-FALLBACK
def load_model_with_fallback(official_id, fallback_id, token=None, trust_remote_code=False):
    try:
        print(f"Trying official repo: {official_id}")
        tok = AutoTokenizer.from_pretrained(official_id, token=token, trust_remote_code=trust_remote_code)
        mdl = AutoModelForCausalLM.from_pretrained(
            official_id, quantization_config=BNB_CONFIG, device_map="auto", token=token,
            trust_remote_code=trust_remote_code
        )
        print(f"Loaded OFFICIAL repo: {official_id}")
        return tok, mdl, official_id
    except Exception as e:
        print(f"Official repo failed ({type(e).__name__}) — falling back to mirror: {fallback_id}")
        tok = AutoTokenizer.from_pretrained(fallback_id, trust_remote_code=trust_remote_code)
        mdl = AutoModelForCausalLM.from_pretrained(fallback_id, quantization_config=BNB_CONFIG,
                                                      device_map="auto", trust_remote_code=trust_remote_code)
        print(f"Loaded FALLBACK repo: {fallback_id}")
        return tok, mdl, fallback_id

print("Model loader ready.")

Model loader ready.


In [6]:
# CELL 5: CYCLIC GAME DEFINITIONS
# Payoff matrix from Park et al. Appendix B.4 (Robinson & Goforth, 2005)
# A = row player payoffs, B = column player payoffs. Actions labeled "1","2" (no cooperate/defect semantics here)
A_MATRIX = {("1","1"): 3, ("1","2"): 1, ("2","1"): 2, ("2","2"): 4}
B_MATRIX = {("1","1"): 3, ("1","2"): 4, ("2","1"): 2, ("2","2"): 1}
ACTIONS = ["1", "2"]

def cyclic_payoffs(row_action, col_action):
    return A_MATRIX[(row_action, col_action)], B_MATRIX[(row_action, col_action)]

# Generalized reactive opponent: best-responds to the model's LAST action (replaces tit-for-tat,
# which only made sense for PD's cooperate/defect framing). Starts at action "1".
def reactive_opponent(history):
    if len(history) == 0:
        return "1"
    last_row_action = history[-1]["your_action"]
    col_payoffs = {c: B_MATRIX[(last_row_action, c)] for c in ACTIONS}
    return max(col_payoffs, key=col_payoffs.get)

# --- Prompt for the REPEATED game (regret sub-experiment) ---
def build_prompt_repeated(history, current_round, total_rounds):
    rules = (
        "You are playing a repeated 2-player game for "
        f"{total_rounds} rounds. Each round, you and your opponent simultaneously choose "
        "either action '1' or action '2'.\n\n"
        "Your payoff depends on both choices:\n"
        "- You choose 1, opponent chooses 1: you get 3\n"
        "- You choose 1, opponent chooses 2: you get 1\n"
        "- You choose 2, opponent chooses 1: you get 2\n"
        "- You choose 2, opponent chooses 2: you get 4\n\n"
        "Your goal is to maximize your own total payoff across all rounds.\n\n"
    )
    if len(history) == 0:
        history_text = "This is round 1. No history yet.\n"
    else:
        history_text = "History so far:\n"
        for i, h in enumerate(history):
            history_text += (
                f"Round {i+1}: You played {h['your_action']}, "
                f"opponent played {h['opponent_action']}, you scored {h['your_payoff']}.\n"
            )
    instruction = (
        f"\nIt is now round {current_round} of {total_rounds}. "
        "Think briefly about your strategy, then respond with your final decision "
        "on a new line in exactly this format:\nDECISION: 1\nor\nDECISION: 2"
    )
    return rules + history_text + instruction

# --- Prompt for the ONE-SHOT game (tau sub-experiment) — NO history, fresh each time ---
def build_prompt_oneshot():
    return (
        "You are playing a single-round 2-player game. You and an opponent will simultaneously "
        "choose either action '1' or action '2'. Your payoff depends on both choices:\n\n"
        "- You choose 1, opponent chooses 1: you get 3\n"
        "- You choose 1, opponent chooses 2: you get 1\n"
        "- You choose 2, opponent chooses 1: you get 2\n"
        "- You choose 2, opponent chooses 2: you get 4\n\n"
        "Your goal is to maximize your own payoff. "
        "Respond with your final decision on a new line in exactly this format:\n"
        "DECISION: 1\nor\nDECISION: 2"
    )

def parse_action_cyclic(response_text):
    match = re.search(r"DECISION:\s*([12])", response_text.upper())
    if match:
        return match.group(1)
    if "2" in response_text[-20:]:
        return "2"
    return "1"

print("Cyclic game logic loaded.")
print("Verified: no pure NE, mixed equilibrium at Row p=0.5 / Col q=0.75 (payoffs 2.5, 2.5)")

Cyclic game logic loaded.
Verified: no pure NE, mixed equilibrium at Row p=0.5 / Col q=0.75 (payoffs 2.5, 2.5)


In [7]:
# CELL 6: CHECKPOINTED REPEATED-GAME ENGINE (REGRET SUB-EXPERIMENT)
def play_cyclic_repeated_checkpointed(model, tokenizer, model_name, total_rounds=10,
                                        max_new_tokens=250, verbose=True):
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{model_name}_cyclic_repeated.json")

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r") as f:
            history = json.load(f)
        print(f"Resuming '{model_name}' cyclic repeated — {len(history)}/{total_rounds} rounds done.")
    else:
        history = []
        print(f"Starting fresh: '{model_name}' cyclic repeated game.")

    for round_num in range(len(history) + 1, total_rounds + 1):
        prompt_text = build_prompt_repeated(history, round_num, total_rounds)
        messages = [
            {"role": "system", "content": "You are a strategic game-playing agent."},
            {"role": "user", "content": prompt_text}
        ]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(model.device)

        t0 = time.time()
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        elapsed = time.time() - t0

        response_text = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        your_action = parse_action_cyclic(response_text)
        opponent_action = reactive_opponent(history)
        your_payoff, opp_payoff = cyclic_payoffs(your_action, opponent_action)

        history.append({
            "round": round_num, "your_action": your_action, "opponent_action": opponent_action,
            "your_payoff": your_payoff, "opponent_payoff": opp_payoff, "full_response": response_text,
        })

        with open(checkpoint_path, "w") as f:
            json.dump(history, f, indent=2)

        rounds_left = total_rounds - round_num
        eta_min = (rounds_left * elapsed) / 60
        if verbose:
            print(f"[{model_name}] Round {round_num}/{total_rounds} | Action: {your_action} | "
                  f"Opp: {opponent_action} | Payoff: {your_payoff} | {elapsed:.0f}s/round | ETA: {eta_min:.1f} min")

    return history

print("Repeated-game engine ready.")

Repeated-game engine ready.


In [8]:
# CELL 7: REGRET + FTRL/FTPL CALCULATORS (GENERALIZED FOR CYCLIC'S "1"/"2" ACTIONS)
def compute_regret_generic(game_history, opponent_history, payoff_fn):
    total_rounds = len(game_history)
    actual_payoff = sum(h["your_payoff"] for h in game_history)
    best_fixed_payoff = max(
        sum(payoff_fn(a, opp)[0] for opp in opponent_history) for a in ACTIONS
    )
    regret = best_fixed_payoff - actual_payoff
    return {
        "actual_payoff": actual_payoff,
        "best_fixed_payoff": best_fixed_payoff,
        "regret_vs_best_fixed": regret,
        "avg_regret_per_round": regret / total_rounds,
    }

def simulate_ftrl_live_generic(opponent_strategy_fn, payoff_fn, total_rounds, learning_rate=0.5, seed=42):
    rng = np.random.default_rng(seed)
    cumulative_scores = {a: 0.0 for a in ACTIONS}
    history, total_payoff = [], 0
    for _ in range(total_rounds):
        scores = np.array([cumulative_scores[a] for a in ACTIONS])
        probs = np.exp(learning_rate * scores) / np.exp(learning_rate * scores).sum()
        chosen = rng.choice(ACTIONS, p=probs)
        opp_act = opponent_strategy_fn(history)
        payoff = payoff_fn(chosen, opp_act)[0]
        total_payoff += payoff
        history.append({"your_action": chosen, "opponent_action": opp_act})
        for a in ACTIONS:
            cumulative_scores[a] += payoff_fn(a, opp_act)[0]
    return {"total_payoff": total_payoff, "actions": [h["your_action"] for h in history]}

def simulate_ftpl_live_generic(opponent_strategy_fn, payoff_fn, total_rounds, noise_scale=1.0, seed=42):
    rng = np.random.default_rng(seed)
    cumulative_scores = {a: 0.0 for a in ACTIONS}
    history, total_payoff = [], 0
    for _ in range(total_rounds):
        noisy_scores = {a: cumulative_scores[a] + rng.normal(0, noise_scale) for a in ACTIONS}
        chosen = max(noisy_scores, key=noisy_scores.get)
        opp_act = opponent_strategy_fn(history)
        payoff = payoff_fn(chosen, opp_act)[0]
        total_payoff += payoff
        history.append({"your_action": chosen, "opponent_action": opp_act})
        for a in ACTIONS:
            cumulative_scores[a] += payoff_fn(a, opp_act)[0]
    return {"total_payoff": total_payoff, "actions": [h["your_action"] for h in history]}

def run_cyclic_regret_evaluation(model, tokenizer, model_name, total_rounds=10, max_new_tokens=250):
    game_history = play_cyclic_repeated_checkpointed(model, tokenizer, model_name, total_rounds, max_new_tokens)
    opponent_actions = [h["opponent_action"] for h in game_history]
    regret_result = compute_regret_generic(game_history, opponent_actions, cyclic_payoffs)
    ftrl_result = simulate_ftrl_live_generic(reactive_opponent, cyclic_payoffs, total_rounds)
    ftpl_result = simulate_ftpl_live_generic(reactive_opponent, cyclic_payoffs, total_rounds)
    return {
        "model": model_name, "game": "cyclic",
        "actual_payoff": regret_result["actual_payoff"],
        "regret_vs_ftrl": ftrl_result["total_payoff"] - regret_result["actual_payoff"],
        "regret_vs_ftpl": ftpl_result["total_payoff"] - regret_result["actual_payoff"],
        "ftrl_payoff": ftrl_result["total_payoff"], "ftpl_payoff": ftpl_result["total_payoff"],
        "action_sequence": [h["your_action"] for h in game_history],
        "raw_history": game_history,
    }

print("Regret/FTRL/FTPL calculators ready (generalized).")

Regret/FTRL/FTPL calculators ready (generalized).


In [9]:
# CELL 8: ONE-SHOT SAMPLING ENGINE (TAU SUB-EXPERIMENT, CHECKPOINTED PER SAMPLE)
def sample_oneshot_checkpointed(model, tokenizer, model_name, n_samples=20, temperature=0.7,
                                  max_new_tokens=250):
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{model_name}_cyclic_oneshot_samples.json")

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r") as f:
            samples = json.load(f)
        print(f"Resuming one-shot sampling '{model_name}' — {len(samples)}/{n_samples} done.")
    else:
        samples = []
        print(f"Starting fresh one-shot sampling: '{model_name}'.")

    prompt_text = build_prompt_oneshot()  # SAME prompt every time — no history, matches literature standard
    messages = [
        {"role": "system", "content": "You are a strategic game-playing agent."},
        {"role": "user", "content": prompt_text}
    ]

    for i in range(len(samples), n_samples):
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(model.device)

        t0 = time.time()
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=temperature, top_p=0.9
        )
        elapsed = time.time() - t0
        response_text = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        action = parse_action_cyclic(response_text)
        samples.append(action)

        with open(checkpoint_path, "w") as f:
            json.dump(samples, f, indent=2)

        print(f"  [{model_name}] one-shot sample {i+1}/{n_samples}: {action} | {elapsed:.0f}s")

    counts = {"1": samples.count("1"), "2": samples.count("2")}
    return {"counts": counts, "n_samples": len(samples)}

print("One-shot sampling engine ready.")

One-shot sampling engine ready.


In [10]:
# CELL 9: LLAMA-3-8B — REGRET + ONE-SHOT TAU EXPERIMENTS
LLAMA_OFFICIAL = "meta-llama/Meta-Llama-3-8B-Instruct"
LLAMA_FALLBACK = "NousResearch/Meta-Llama-3-8B-Instruct"

if "model_llama" not in globals():
    tokenizer_llama, model_llama, llama_id_used = load_model_with_fallback(
        LLAMA_OFFICIAL, LLAMA_FALLBACK, token=os.environ.get("HF_TOKEN")
    )

result_llama_regret = run_cyclic_regret_evaluation(model_llama, tokenizer_llama, "Llama-3-8B-Instruct",
                                                     total_rounds=10, max_new_tokens=250)
print(f"Llama-3-8B Cyclic regret vs FTRL: {result_llama_regret['regret_vs_ftrl']:+d}")

result_llama_tau = sample_oneshot_checkpointed(model_llama, tokenizer_llama, "Llama-3-8B-Instruct",
                                                 n_samples=20, temperature=0.7, max_new_tokens=250)
print(f"Llama-3-8B one-shot counts: {result_llama_tau['counts']}")

Trying official repo: meta-llama/Meta-Llama-3-8B-Instruct


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Loaded OFFICIAL repo: meta-llama/Meta-Llama-3-8B-Instruct
Resuming 'Llama-3-8B-Instruct' cyclic repeated — 10/10 rounds done.
Llama-3-8B Cyclic regret vs FTRL: +5
Resuming one-shot sampling 'Llama-3-8B-Instruct' — 20/20 done.
Llama-3-8B one-shot counts: {'1': 0, '2': 20}


In [11]:
# CELL 10: DEEPSEEK-R1-DISTILL — REGRET + ONE-SHOT TAU EXPERIMENTS
if "model_llama" in globals():
    del model_llama, tokenizer_llama
    gc.collect()
    torch.cuda.empty_cache()

R1_OFFICIAL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"

if "model_r1" not in globals():
    tokenizer_r1, model_r1, r1_id_used = load_model_with_fallback(
        R1_OFFICIAL, R1_OFFICIAL, token=os.environ.get("HF_TOKEN")
    )

# max_new_tokens cited: Jia et al. 2025 Table 8, mixed-motive category — mean 2318, max 3742
result_r1_regret = run_cyclic_regret_evaluation(model_r1, tokenizer_r1, "DeepSeek-R1-Distill-Qwen-14B",
                                                  total_rounds=10, max_new_tokens=4000)
print(f"DeepSeek-R1 Cyclic regret vs FTRL: {result_r1_regret['regret_vs_ftrl']:+d}")

# n_samples reduced to 8 for R1 specifically — same time-budget reasoning as Day 4 (R1 generations are slow)
result_r1_tau = sample_oneshot_checkpointed(model_r1, tokenizer_r1, "DeepSeek-R1-Distill-Qwen-14B",
                                              n_samples=8, temperature=0.7, max_new_tokens=4000)
print(f"DeepSeek-R1 one-shot counts: {result_r1_tau['counts']}")

Trying official repo: deepseek-ai/DeepSeek-R1-Distill-Qwen-14B


config.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Loaded OFFICIAL repo: deepseek-ai/DeepSeek-R1-Distill-Qwen-14B
Resuming 'DeepSeek-R1-Distill-Qwen-14B' cyclic repeated — 10/10 rounds done.
DeepSeek-R1 Cyclic regret vs FTRL: -4
Resuming one-shot sampling 'DeepSeek-R1-Distill-Qwen-14B' — 8/8 done.
DeepSeek-R1 one-shot counts: {'1': 0, '2': 8}


In [12]:
# CELL 10B: QWEN3-8B — REGRET + ONE-SHOT TAU EXPERIMENTS
if "model_r1" in globals():
    del model_r1, tokenizer_r1
    gc.collect()
    torch.cuda.empty_cache()

QWEN_OFFICIAL = "Qwen/Qwen3-8B"

if "model_qwen" not in globals():
    tokenizer_qwen, model_qwen, qwen_id_used = load_model_with_fallback(
        QWEN_OFFICIAL, QWEN_OFFICIAL, token=os.environ.get("HF_TOKEN")
    )

# Qwen3 has thinking mode ON by default via its chat template — treating it as a reasoning-tier
# model, using the same generous token budget as R1 (approximate, applied consistently, not a
# per-model citation — same logic as Day 3's R1 setting).
result_qwen_regret = run_cyclic_regret_evaluation(model_qwen, tokenizer_qwen, "Qwen3-8B",
                                                    total_rounds=10, max_new_tokens=4000)
print(f"Qwen3-8B Cyclic regret vs FTRL: {result_qwen_regret['regret_vs_ftrl']:+d}")

result_qwen_tau = sample_oneshot_checkpointed(model_qwen, tokenizer_qwen, "Qwen3-8B",
                                                n_samples=8, temperature=0.7, max_new_tokens=4000)
print(f"Qwen3-8B one-shot counts: {result_qwen_tau['counts']}")

Trying official repo: Qwen/Qwen3-8B


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded OFFICIAL repo: Qwen/Qwen3-8B
Resuming 'Qwen3-8B' cyclic repeated — 10/10 rounds done.
Qwen3-8B Cyclic regret vs FTRL: +2
Resuming one-shot sampling 'Qwen3-8B' — 8/8 done.
Qwen3-8B one-shot counts: {'1': 1, '2': 7}


In [13]:
# CELL 10C: GLM-4-9B-CHAT — REGRET + ONE-SHOT TAU EXPERIMENTS
if "model_qwen" in globals():
    del model_qwen, tokenizer_qwen
    gc.collect()
    torch.cuda.empty_cache()

GLM_OFFICIAL = "zai-org/glm-4-9b-chat"

if "model_glm" not in globals():
    tokenizer_glm, model_glm, glm_id_used = load_model_with_fallback(
        GLM_OFFICIAL, GLM_OFFICIAL, token=os.environ.get("HF_TOKEN"), trust_remote_code=True
    )

result_glm_regret = run_cyclic_regret_evaluation(model_glm, tokenizer_glm, "GLM-4-9B-Chat",
                                                    total_rounds=10, max_new_tokens=250)
print(f"GLM-4-9B-Chat Cyclic regret vs FTRL: {result_glm_regret['regret_vs_ftrl']:+d}")

result_glm_tau = sample_oneshot_checkpointed(model_glm, tokenizer_glm, "GLM-4-9B-Chat",
                                               n_samples=20, temperature=0.7, max_new_tokens=250)
print(f"GLM-4-9B-Chat one-shot counts: {result_glm_tau['counts']}")

Trying official repo: zai-org/glm-4-9b-chat


config.json: 0.00B [00:00, ?B/s]

configuration_chatglm.py: 0.00B [00:00, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/zai-org/glm-4-9b-chat:
- configuration_chatglm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Official repo failed (AttributeError) — falling back to mirror: zai-org/glm-4-9b-chat


[transformers] ChatGLMForConditionalGeneration has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


AttributeError: 'ChatGLMConfig' object has no attribute 'max_length'

In [ ]:
# CELL 11: TAU FITTING — CORRECTED COGNITIVE HIERARCHY AGGREGATION
# (fit_tau_ch_corrected function definition stays exactly as-is — unchanged)

tau_llama, nll_fn_llama = fit_tau_ch_corrected(result_llama_tau["counts"], cyclic_payoffs)
tau_r1, nll_fn_r1 = fit_tau_ch_corrected(result_r1_tau["counts"], cyclic_payoffs)
tau_qwen, nll_fn_qwen = fit_tau_ch_corrected(result_qwen_tau["counts"], cyclic_payoffs)
tau_glm, nll_fn_glm = fit_tau_ch_corrected(result_glm_tau["counts"], cyclic_payoffs)

print(f"Llama-3-8B     | Cyclic regret vs FTRL: {result_llama_regret['regret_vs_ftrl']:+d} | τ: {tau_llama:.2f} (n={result_llama_tau['n_samples']})")
print(f"DeepSeek-R1    | Cyclic regret vs FTRL: {result_r1_regret['regret_vs_ftrl']:+d} | τ: {tau_r1:.2f} (n={result_r1_tau['n_samples']})")
print(f"Qwen3-8B       | Cyclic regret vs FTRL: {result_qwen_regret['regret_vs_ftrl']:+d} | τ: {tau_qwen:.2f} (n={result_qwen_tau['n_samples']})")
print(f"GLM-4-9B-Chat  | Cyclic regret vs FTRL: {result_glm_regret['regret_vs_ftrl']:+d} | τ: {tau_glm:.2f} (n={result_glm_tau['n_samples']})")

In [ ]:
# CELL 12: NLL DIAGNOSTIC — CONFIRM GENUINE MINIMUM, NOT BOUNDARY COLLAPSE
import matplotlib.pyplot as plt

tau_range = np.linspace(0.01, 8.0, 200)
nll_llama_curve = np.array([nll_fn_llama(t) for t in tau_range])
nll_r1_curve = np.array([nll_fn_r1(t) for t in tau_range])
nll_qwen_curve = np.array([nll_fn_qwen(t) for t in tau_range])
nll_glm_curve = np.array([nll_fn_glm(t) for t in tau_range])

plt.figure(figsize=(10, 5))
plt.plot(tau_range, nll_llama_curve, label="Llama-3-8B", linewidth=2)
plt.plot(tau_range, nll_r1_curve, label="DeepSeek-R1-Distill", linewidth=2)
plt.plot(tau_range, nll_qwen_curve, label="Qwen3-8B", linewidth=2)
plt.plot(tau_range, nll_glm_curve, label="GLM-4-9B-Chat", linewidth=2)
plt.xlabel("τ"); plt.ylabel("NLL (lower = better)")
plt.title("Cyclic game — NLL curve, 4 models")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(OUTPUT_DIR, "cyclic_nll_diagnostic_4models.png"), dpi=100, bbox_inches='tight')
plt.show()

for name, curve in [("Llama", nll_llama_curve), ("R1", nll_r1_curve), ("Qwen3", nll_qwen_curve), ("GLM-4", nll_glm_curve)]:
    print(f"{name} min NLL: {curve.min():.3f} at τ={tau_range[np.argmin(curve)]:.2f} | boundary (0.01): {curve[0]:.3f}")

In [ ]:
# CELL 13: SAVE ALL RESULTS TO OUTPUT (DOWNLOADABLE) — FIXED TUPLE-KEY BUG, ALL 4 MODELS
def result_block(regret_result, tau_result, tau_value):
    return {
        "regret_vs_ftrl": regret_result["regret_vs_ftrl"],
        "regret_vs_ftpl": regret_result["regret_vs_ftpl"],
        "action_sequence_repeated": regret_result["action_sequence"],
        "oneshot_counts": tau_result["counts"],
        "tau": float(tau_value),
    }

final_summary = {
    "game": "cyclic",
    "matrix": {
        "A": {f"{k[0]},{k[1]}": v for k, v in A_MATRIX.items()},
        "B": {f"{k[0]},{k[1]}": v for k, v in B_MATRIX.items()},
    },
    "equilibrium": {"pure_NE": None, "mixed_NE": {"row_p": 0.5, "col_q": 0.75, "payoff": 2.5}},
    "results": {
        "Llama-3-8B-Instruct": result_block(result_llama_regret, result_llama_tau, tau_llama),
        "DeepSeek-R1-Distill-Qwen-14B": result_block(result_r1_regret, result_r1_tau, tau_r1),
        "Qwen3-8B": result_block(result_qwen_regret, result_qwen_tau, tau_qwen),
        "GLM-4-9B-Chat": result_block(result_glm_regret, result_glm_tau, tau_glm),
    }
}

output_path = os.path.join(OUTPUT_DIR, "cyclic_game_final_results_4models.json")
with open(output_path, "w") as f:
    json.dump(final_summary, f, indent=2)

print(f"Saved final results to: {output_path}")
print(json.dumps(final_summary, indent=2))